[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ohtopico/site/blob/main/notebooks/wikipedia/04-sazonalidade-escolar.ipynb)

# Sazonalidade escolar: o calendário que aparece nos dados

Verbetes como Fração e Fotossíntese são lidos por estudantes. Por isso, as visitas sobem no ano letivo, caem nas férias e diminuem nos fins de semana. Neste notebook você encontra esse padrão nos dados e compara com verbetes de lazer.

**Público:** professores, 6º ao 9º ano e ensino médio · **Tempo:** 1 aula de 50 min · **BNCC:** EF06MA32, EF07MA35, EF07MA36, EF09MA22

Notebook de [Júlia Roberta Quoos Alves](https://ohtopico.com.br/curriculo/) · [ohtopico.com.br](https://ohtopico.com.br/interesse-por-ia/notebooks/) · Licença CC BY-NC-SA 4.0

**Como usar:** clique em cada célula de código e aperte o botão de play (ou Shift + Enter), de cima para baixo. Os campos ao lado das células (listas e caixas de texto) podem ser trocados à vontade: depois de mudar, rode a célula de novo.

## Objetivos
- Reconhecer padrões que se repetem (sazonalidade) em uma série temporal.
- Comparar perfis com números de tamanhos diferentes usando um índice (média = 100).
- Ler um mapa de calor.

In [ ]:
# @title Configuração (rode primeiro) { display-mode: "form" }
# Funções para baixar os dados públicos da Wikimedia. Não precisa mudar nada aqui.
import json, re, time, urllib.parse
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets

CORES = ['#157f3b', '#b3123a', '#c99a00', '#111111', '#5fb27c', '#7a1f33']
plt.rcParams.update({'figure.figsize': (11, 4.5), 'axes.prop_cycle': plt.cycler(color=CORES), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .3, 'font.size': 11})
pd.options.display.float_format = '{:,.0f}'.format
UA = {'User-Agent': 'notebooks-ohtopico/1.0 (https://ohtopico.com.br; contato@ohtopico.com.br)'}
FONTE = 'Fonte: Wikimedia, visitas de pessoas à Wikipédia em português'


def _get(url, params=None):
    r = requests.get(url, params=params, headers=UA, timeout=60)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()


def titulo_oficial(titulo):
    """Corrige maiúsculas e segue redirecionamentos: 'ia' vira 'Inteligência artificial'."""
    j = _get('https://pt.wikipedia.org/w/api.php', dict(action='query', titles=titulo, redirects=1, format='json', formatversion=2))
    pagina = j['query']['pages'][0]
    if pagina.get('missing'):
        raise ValueError(f'Não achei o verbete "{titulo}" na Wikipédia em português. Confira a grafia.')
    return pagina['title']


def visitas(titulos, inicio='2016-01-01', fim=None, por='mes'):
    """Tabela com as visitas de cada verbete. por='mes' ou por='dia'."""
    if isinstance(titulos, str):
        titulos = [titulos]
    fim = pd.Timestamp(fim) if fim else pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    if por == 'mes':
        fim = fim.replace(day=1) - pd.Timedelta(days=1)  # só meses completos
    colunas = {}
    for t in titulos:
        t = titulo_oficial(t)
        art = urllib.parse.quote(t.replace(' ', '_'), safe='')
        g = 'monthly' if por == 'mes' else 'daily'
        url = (f'https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/pt.wikipedia/all-access/user/{art}/{g}/'
               f'{pd.Timestamp(inicio):%Y%m%d}00/{fim:%Y%m%d}00')
        j = _get(url) or {'items': []}
        colunas[t] = pd.Series({pd.Timestamp(i['timestamp'][:8]): i['views'] for i in j['items']}, dtype='float64')
        time.sleep(.1)
    df = pd.DataFrame(colunas).sort_index()
    idx = pd.date_range(pd.Timestamp(inicio), fim, freq='MS' if por == 'mes' else 'D')
    df = df.reindex(idx).fillna(0).astype(int)
    df.index.name = 'mês' if por == 'mes' else 'dia'
    return df


def texto(titulo):
    """Texto puro de um verbete (sem tabelas nem imagens)."""
    t = titulo_oficial(titulo)
    j = _get('https://pt.wikipedia.org/w/api.php', dict(action='query', prop='extracts', explaintext=1, titles=t,
                                                         format='json', formatversion=2))
    return j['query']['pages'][0].get('extract', '')


def fmt(v, casas=0):
    """Número no formato brasileiro: 1.234,5"""
    return f'{v:,.{casas}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


def baixar_csv(df, nome):
    df.to_csv(nome)
    try:
        from google.colab import files
        files.download(nome)
    except ImportError:
        print('Arquivo salvo:', nome)


print('Pronto. Funções disponíveis: visitas(), texto(), titulo_oficial(), fmt(), baixar_csv().')

## Passo 1. Verbetes de escola e de lazer

In [ ]:
escola = ['Fração', 'Fotossíntese', 'Teorema de Pitágoras', 'Revolução Francesa', 'Célula']
lazer = ['Minecraft', 'Roblox', 'Carnaval']
meses = visitas(escola + lazer, inicio='2018-01-01')
meses.tail(3)

## Passo 2. O perfil de cada mês
Cada verbete tem um tamanho diferente. Para comparar, dividimos cada mês pela média do próprio ano e multiplicamos por 100: acima de 100, o mês foi mais lido que a média daquele ano.

In [ ]:
MESES = ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']
completos = meses[meses.index.year < pd.Timestamp.today().year]
indice = completos / completos.groupby(completos.index.year).transform('mean') * 100
perfil = indice.groupby(indice.index.month).mean()
perfil.index = MESES
ax = perfil[escola].mean(axis=1).plot(linewidth=3, marker='o', label='média dos verbetes de escola')
perfil[lazer].mean(axis=1).plot(ax=ax, linewidth=3, marker='o', label='média dos verbetes de lazer')
ax.axhline(100, color='#111', linewidth=1, linestyle='--')
ax.set_xticks(range(12)); ax.set_xticklabels(MESES)
ax.set_title('Perfil do ano (100 = média do ano)'); ax.legend(); plt.show()

**Para discutir:** em que meses os verbetes de escola ficam abaixo de 100? Isso combina com o calendário escolar brasileiro? E o Carnaval, quando sobe?

## Passo 3. Mapa de calor
Cada linha é um ano e cada coluna é um mês. Quanto mais escuro o verde, mais visitas em relação à média daquele ano.

In [ ]:
@interact(verbete=escola + lazer)
def mapa(verbete):
    tabela = indice[verbete].groupby([indice.index.year, indice.index.month]).mean().unstack()
    fig, ax = plt.subplots(figsize=(11, .5 * len(tabela) + 1.5))
    im = ax.imshow(tabela, cmap='YlGn', aspect='auto')
    ax.set_xticks(range(12)); ax.set_xticklabels(MESES)
    ax.set_yticks(range(len(tabela))); ax.set_yticklabels(tabela.index)
    for i in range(tabela.shape[0]):
        for j in range(tabela.shape[1]):
            v = tabela.iloc[i, j]
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=9, color='white' if v > 120 else '#111')
    ax.grid(False); ax.set_title(f'{verbete}: visitas em relação à média do ano (100)')
    plt.colorbar(im, ax=ax, shrink=.8); plt.show()

## Passo 4. Dias da semana
Agora com dados por dia: em que dia da semana cada verbete é mais lido?

In [ ]:
DIAS = ['seg', 'ter', 'qua', 'qui', 'sex', 'sáb', 'dom']
por_dia = visitas(['Fração', 'Minecraft'], inicio=pd.Timestamp.today() - pd.Timedelta(days=365), por='dia')
semana = por_dia.groupby(por_dia.index.dayofweek).mean()
semana = semana / semana.mean() * 100
semana.index = DIAS
ax = semana.plot(kind='bar', width=.8, edgecolor='#111')
ax.axhline(100, color='#111', linewidth=1, linestyle='--')
ax.set_title('Visitas por dia da semana (100 = média da semana)'); plt.xticks(rotation=0); plt.show()

## Sua vez
Escolha um verbete ligado a uma data do ano (por exemplo, *Dia das Crianças*, *Páscoa* ou *Revolução Farroupilha*) e descubra o mês de pico usando o perfil do Passo 2.

In [ ]:
# escreva aqui

<details><summary><b>Ver uma solução</b></summary>

```python
v = visitas('Revolução Farroupilha', inicio='2018-01-01')
v = v[v.index.year < pd.Timestamp.today().year]
perfil_v = (v / v.groupby(v.index.year).transform('mean') * 100).groupby(v.index.month).mean()
print('Mês de pico:', MESES[perfil_v.iloc[:, 0].idxmax() - 1])
```
</details>

## Para fechar
Saber que um padrão se repete ajuda a não confundir sazonalidade com tendência: uma queda em julho não quer dizer que o interesse diminuiu, só que é férias. Por isso, para comparar, use o mesmo mês do ano anterior.